# Fine-tune **Qwen2.5-VL-7B** for TTB Label Verification

Self-contained Colab notebook. LoRA fine-tune on `unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit` using Unsloth. Trains on data pulled from the COLA Cloud free sample (CC0, ~1700 image-JSON pairs).

## What this notebook does
1. Validates the install (fails fast on any ABI / version mismatch)
2. Mounts Google Drive at `/MyDrive/ttb_sft/{model_tag}/`
3. Downloads the COLA Cloud sample + ~800 label images
4. Builds (image, structured-JSON) training pairs matching the backend's `ExtractedLabel` schema
5. 80/10/10 train/val/test split (seeded for reproducibility)
6. Loads the base model in 4-bit + applies a LoRA adapter
7. **Smoke test**: 1 training step on 4 samples (catches bugs in <30s before committing GPU hours)
8. Full LoRA fine-tune (2 epochs)
9. Per-field accuracy eval on the held-out test set
10. Auto-zip + browser-download the adapter to your Mac's `~/Downloads`

## Runtime recommendation
**A100 (Colab Pro)** is recommended — full 2-epoch run is ~3 hours. T4 will OOM on the 7B model; if you only have T4 access, use the Kimi notebook instead.

## Before you Run All
**If you ran a previous SFT session on this VM**, do `Runtime → Disconnect and delete runtime` first — that clears any cached numpy/transformers state from a prior install. Then `Runtime → Run all`. The notebook is fully automated; caffeinate your Mac (`caffeinate -i &`) and walk away.


## 0. Install dependencies


In [ ]:
# Qwen2.5-VL via Unsloth — install Unsloth (it manages compatible torch /
# transformers / bitsandbytes versions automatically) + TRL for the SFT trainer.
!pip install -q --upgrade pip
!pip install -q --upgrade 'unsloth' 'trl<0.10.0' peft accelerate 'bitsandbytes>=0.44'
!pip install -q --upgrade datasets pillow requests tqdm


In [ ]:
MODEL_TAG = 'qwen2_5_vl_7b'
BASE_MODEL = 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'


## 0.5. Validate imports (fails fast on ABI mismatch)

If this cell raises `numpy.dtype size changed` or a similar binary-incompatibility error, do `Runtime → Disconnect and delete runtime` (fresh VM, not just restart) and Run all again.


In [ ]:
import importlib
_required = ['torch', 'transformers', 'accelerate', 'peft', 'trl', 'bitsandbytes', 'datasets', 'PIL', 'tqdm', 'unsloth']
_failed = []
for mod in _required:
    try:
        m = importlib.import_module(mod)
        print(f'  ✓ {mod:<14} {getattr(m, "__version__", "?")}')
    except Exception as e:
        print(f'  ✗ {mod:<14} FAILED — {type(e).__name__}: {e}')
        _failed.append(mod)
if _failed:
    raise RuntimeError(
        f'{len(_failed)} import(s) failed: {_failed}. Most common cause: numpy ABI '
        f'mismatch — do Runtime → Disconnect and delete runtime, then Run all.'
    )
print('\n✅ All imports clean.')


## 1. Mount Drive + set up work directories


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
WORK_DIR = Path(f'/content/drive/MyDrive/ttb_sft/{MODEL_TAG}')
WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path('/content/cola_data'); DATA_DIR.mkdir(exist_ok=True)
IMG_DIR  = DATA_DIR / 'images';        IMG_DIR.mkdir(exist_ok=True)
print('Work dir:', WORK_DIR)


## 2. Download COLA Cloud free sample (CC0)

1,000 records + ~1,750 image rows. ~500 KB zip. Sourced from the official TTB Public COLA Registry, packaged for direct fetch.


In [ ]:
import urllib.request, zipfile
COLA_ZIP_URL = 'https://dyuie4zgfxmt6.cloudfront.net/samples/cola-sample-pack-v1.zip'
zip_path = DATA_DIR / 'sample.zip'
if not (DATA_DIR / 'cola.csv').exists():
    print('Downloading COLA Cloud sample...')
    urllib.request.urlretrieve(COLA_ZIP_URL, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print('  Extracted to', DATA_DIR)
else:
    print('Already downloaded.')
!ls -la {DATA_DIR}


## 3. Build (image, structured-JSON) training pairs

For each label image with usable metadata, synthesize the structured JSON the model should learn. The schema matches the backend's `ExtractedLabel` shape so the SFT'd adapter is drop-in for `INFERENCE_MODE=sft`.

Ground-truth sources from the dataset:
- `BRAND_NAME`, `PRODUCT_NAME`, `CLASS_NAME`, `ORIGIN_NAME` — COLA form fields
- `OCR_ABV`, `OCR_VOLUME` — Google Vision OCR on the label
- `OCR_TEXT` — per-image OCR (for warning + bottler line extraction)


In [ ]:
import csv, json, re
from collections import defaultdict

def _net_str(row):
    v = (row.get('OCR_VOLUME') or '').strip()
    u = (row.get('OCR_VOLUME_UNIT') or '').strip().lower()
    if not v: return ''
    try:
        f = float(v); v = str(int(f)) if f.is_integer() else str(f)
    except ValueError: pass
    unit_map = {'milliliters':'mL','milliliter':'mL','ml':'mL',
                'liters':'L','liter':'L','l':'L',
                'fluid ounces':'FL OZ','fluid ounce':'FL OZ','fl oz':'FL OZ','fl. oz.':'FL OZ',
                'ounces':'FL OZ','ounce':'FL OZ','oz':'FL OZ'}
    return f"{v} {unit_map.get(u, u or 'mL')}"

def _abv_str(row):
    v = (row.get('OCR_ABV') or '').strip()
    if not v: return ''
    try:
        f = float(v); return f'{f:g}% Alc./Vol.'
    except ValueError: return ''

def _extract_warning(ocr_text):
    if not ocr_text or 'GOVERNMENT WARNING' not in ocr_text.upper(): return None
    start = ocr_text.upper().find('GOVERNMENT WARNING')
    tail = ocr_text[start:]
    m = re.search(r'machinery[^.]*\.\s*', tail, re.IGNORECASE)
    return tail[:m.end()].strip() if m else tail[:500].strip()

def _extract_bottler(ocr_text):
    if not ocr_text: return ''
    for kw in ('BOTTLED BY','BREWED BY','PRODUCED BY','DISTILLED BY','IMPORTED BY',
               'Bottled by','Brewed by','Produced by','Distilled by','Imported by'):
        if kw in ocr_text:
            idx = ocr_text.find(kw); end = ocr_text.find('.', idx)
            return ocr_text[idx:end if end != -1 else idx + 200].strip()
    return ''

def _derive_beverage(row):
    pt = (row.get('PRODUCT_TYPE') or '').lower(); cn = (row.get('CLASS_NAME') or '').lower()
    if 'seltzer' in cn or 'flavored' in cn: return 'malt'
    if pt == 'distilled spirits': return 'spirits'
    if pt == 'wine': return 'wine'
    if pt == 'malt beverage': return 'beer'
    return 'wine'

cola = list(csv.DictReader(open(DATA_DIR / 'cola.csv')))
img_rows = list(csv.DictReader(open(DATA_DIR / 'cola_image.csv')))
by_ttb = defaultdict(list)
for ir in img_rows: by_ttb[ir['TTB_ID']].append(ir)

candidates = []
for cola_row in cola:
    for img_row in by_ttb.get(cola_row['TTB_ID'], []):
        ocr_text = img_row.get('OCR_TEXT') or ''
        if not ocr_text: continue
        warning_text = _extract_warning(ocr_text)
        pos = img_row.get('CONTAINER_POSITION', '').upper()
        is_back = pos == 'BACK'
        target = {
            'fields': {},
            'government_warning': {
                'present': warning_text is not None,
                'detected_text': warning_text or '',
                'casing_all_caps': bool(warning_text and warning_text[:18].isupper()),
                'heading_bold': bool(warning_text),
                'body_bold': False,
                'approx_font_mm': None,
                'contrast_ok': True, 'separate_and_apart': True,
            },
            'image_quality': {'score': 0.85, 'legible': True, 'note': None},
        }
        if not is_back:
            if cola_row.get('BRAND_NAME'):
                target['fields']['Brand name'] = {'value': cola_row['BRAND_NAME'], 'confidence': 0.95}
            if cola_row.get('CLASS_NAME'):
                target['fields']['Class & type'] = {'value': cola_row['CLASS_NAME'], 'confidence': 0.95}
        if _abv_str(cola_row) and img_row.get('TTB_IMAGE_ID') == cola_row.get('OCR_ABV_TTB_IMAGE_ID'):
            target['fields']['Alcohol content'] = {'value': _abv_str(cola_row), 'confidence': 0.92}
        if _net_str(cola_row) and img_row.get('TTB_IMAGE_ID') == cola_row.get('OCR_VOLUME_TTB_IMAGE_ID'):
            target['fields']['Net contents'] = {'value': _net_str(cola_row), 'confidence': 0.92}
        bottler = _extract_bottler(ocr_text)
        if bottler and is_back:
            target['fields']['Bottler name/address'] = {'value': bottler, 'confidence': 0.85}
        if (cola_row.get('DOMESTIC_OR_IMPORTED') or '').lower() == 'imported' and cola_row.get('ORIGIN_NAME') and is_back:
            target['fields']['Country of origin'] = {'value': f"Product of {cola_row['ORIGIN_NAME'].title()}", 'confidence': 0.85}
        if not target['fields'] and not target['government_warning']['present']:
            continue
        candidates.append({
            'ttb_image_id': img_row['TTB_IMAGE_ID'],
            'beverage_type': _derive_beverage(cola_row),
            'target': target,
        })

print(f'Built {len(candidates)} candidate (image, JSON) pairs')
print(f'Sample target:\n{json.dumps(candidates[0]["target"], indent=2)[:500]}')


## 4. Download label images (parallel, ~3 min)


In [ ]:
import concurrent.futures, urllib.request
CDN = 'https://dyuie4zgfxmt6.cloudfront.net/{}.webp'

def _dl(cand):
    dest = IMG_DIR / f"{cand['ttb_image_id']}.webp"
    if dest.exists() and dest.stat().st_size > 0: return cand, True
    try:
        req = urllib.request.Request(CDN.format(cand['ttb_image_id']),
                                     headers={'User-Agent': 'ttb-sft/0.1'})
        with urllib.request.urlopen(req, timeout=20) as r:
            dest.write_bytes(r.read())
        return cand, True
    except Exception:
        return cand, False

downloaded = []
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as ex:
    for cand, ok in ex.map(_dl, candidates):
        if ok:
            cand['image_path'] = str(IMG_DIR / f"{cand['ttb_image_id']}.webp")
            downloaded.append(cand)
print(f'Downloaded {len(downloaded)} / {len(candidates)} images')


## 5. Train / val / test split (80 / 10 / 10)


In [ ]:
import random, json
random.seed(42); random.shuffle(downloaded)
n = len(downloaded); n_train = int(n * 0.80); n_val = int(n * 0.10)
train = downloaded[:n_train]
val   = downloaded[n_train:n_train + n_val]
test  = downloaded[n_train + n_val:]
print(f'train={len(train)}  val={len(val)}  test={len(test)}')

(WORK_DIR / 'splits').mkdir(exist_ok=True)
for name, split in [('train', train), ('val', val), ('test', test)]:
    with open(WORK_DIR / 'splits' / f'{name}.jsonl', 'w') as f:
        for ex in split:
            f.write(json.dumps(ex) + '\n')
    print(f'  wrote splits/{name}.jsonl')


In [ ]:
SYSTEM_PROMPT = (
    'You are a careful transcription assistant for U.S. TTB alcohol label review. '
    'Given ONE label panel image and the beverage type, READ what is printed and '
    'RETURN ONLY a JSON object matching the schema below. Do NOT decide compliance.\n\n'
    'If a field is not clearly visible, OMIT it from the fields object. Never guess; '
    'never substitute deposit codes (e.g. "CA CRV"), NOM IDs, or barcodes. Transcribe '
    'the Government Warning EXACTLY as printed (preserve case, punctuation, errors).\n\n'
    'Schema: {fields: {<field name>: {value, confidence}}, government_warning: '
    '{present, detected_text, casing_all_caps, heading_bold, body_bold, approx_font_mm, '
    'contrast_ok, separate_and_apart}, image_quality: {score, legible, note}}'
)


## 6. Load Qwen2.5-VL-7B (pre-quantized 4-bit) + apply LoRA

Uses Unsloth's pre-quantized `unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit` — same weights as the upstream model but ships in 4-bit so the Colab download is faster.


In [ ]:
import torch
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit',
    load_in_4bit = True,
    use_gradient_checkpointing = 'unsloth',
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # vision tower frozen (saves memory; fine for OCR-like task)
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = 'none',
    random_state = 42,
)
model.print_trainable_parameters()
FastVisionModel.for_training(model)  # explicit training mode


## 7. Format dataset for SFT (Unsloth conversation schema)


In [ ]:
from PIL import Image
import json

def _convo(ex):
    return {'messages': [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user',   'content': [
            {'type': 'image', 'image': Image.open(ex['image_path']).convert('RGB')},
            {'type': 'text',  'text': f"Beverage type: {ex['beverage_type']}. Extract per the schema."},
        ]},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': json.dumps(ex['target'])}]},
    ]}

train_ds = [_convo(ex) for ex in train]
print(f'train examples: {len(train_ds)}')


## 7.5. Smoke test — 1 training step on 4 samples

**This cell runs in ~30 seconds and catches almost every bug** (broken imports, OOM, model-config mismatches, data-collator issues) BEFORE you commit to the full ~2-hour training run. If it crashes, fix the issue HERE — don't proceed.


In [ ]:
from trl import SFTTrainer, SFTConfig

# Subset the dataset for a single training step.
if hasattr(train_ds, 'select'):   # HF Dataset (Kimi path)
    _smoke = train_ds.select(range(min(4, len(train_ds))))
else:                               # list of dicts (Qwen path)
    _smoke = list(train_ds)[:4]

try:
    import torch
    _bf16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
    _kw = {
        'model': model,
        'train_dataset': _smoke,
        'args': SFTConfig(
            per_device_train_batch_size = 1,
            gradient_accumulation_steps = 1,
            max_steps = 1,
            warmup_steps = 0,
            learning_rate = 1e-4,
            fp16 = not _bf16, bf16 = _bf16,
            logging_steps = 1,
            output_dir = '/tmp/smoke',
            report_to = 'none',
            remove_unused_columns = False,
            max_seq_length = 2048,
            dataset_kwargs = {'skip_prepare_dataset': True},
        ),
    }
    # Unsloth's collator OR our Kimi _collate — whichever is defined.
    try:
        from unsloth.trainer import UnslothVisionDataCollator
        _kw['data_collator'] = UnslothVisionDataCollator(model, tokenizer)
        _kw['tokenizer'] = tokenizer
    except ImportError:
        _kw['data_collator'] = _collate
    SFTTrainer(**_kw).train()
    print('\n✅ SMOKE TEST PASSED — full training run is safe to start.')
except Exception:
    import traceback; traceback.print_exc()
    raise RuntimeError('Smoke test failed — fix the issue above before running full training.')


## 8. Full training run (2 epochs)


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_ds,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = False, bf16 = True,
        logging_steps = 10,
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        lr_scheduler_type = 'linear',
        seed = 42,
        output_dir = str(WORK_DIR / 'checkpoints'),
        save_strategy = 'epoch',
        report_to = 'none',
        remove_unused_columns = False,
        dataset_kwargs = {'skip_prepare_dataset': True},
        max_seq_length = 2048,
    ),
)
trainer.train()


## 9.5. Per-field scoring helper


In [ ]:
def _run_eval(test_set, gen_fn):
    import json
    from collections import Counter
    from tqdm.auto import tqdm
    field_correct = Counter(); field_total = Counter()
    warn_match = 0; warn_total = 0
    n_parsed = n_unparseable = 0
    for ex in tqdm(test_set, desc='Evaluating'):
        pred = gen_fn(ex)
        if pred is None:
            n_unparseable += 1; continue
        n_parsed += 1
        for fname, fobs in (ex['target']['fields'] or {}).items():
            field_total[fname] += 1
            got = ((pred.get('fields') or {}).get(fname) or {}).get('value', '')
            if _norm(got) == _norm(fobs['value']):
                field_correct[fname] += 1
        warn_total += 1
        if bool(ex['target']['government_warning']['present']) == bool((pred.get('government_warning') or {}).get('present')):
            warn_match += 1
    print(f'\nParsed: {n_parsed} / Unparseable: {n_unparseable}')
    print('Per-field accuracy:')
    for f in sorted(field_total):
        acc = field_correct[f] / field_total[f]
        print(f'  {f:<24} {field_correct[f]:>3} / {field_total[f]:>3} = {acc:.1%}')
    if warn_total:
        print(f'Warning-presence accuracy: {warn_match}/{warn_total} = {warn_match/warn_total:.1%}')
    report = {
        'n_test': len(test_set), 'n_parsed': n_parsed, 'n_unparseable': n_unparseable,
        'field_accuracy': {f: {'correct': field_correct[f], 'total': field_total[f]} for f in field_total},
        'warning_present_accuracy': (warn_match / warn_total) if warn_total else None,
    }
    with open(WORK_DIR / 'eval_report.json', 'w') as f:
        json.dump(report, f, indent=2)
    print(f'\nReport → {WORK_DIR}/eval_report.json')


## 9. Evaluate on held-out test set (per-field accuracy)


In [ ]:
import json, re, torch
from collections import Counter
from PIL import Image
from unsloth import FastVisionModel
FastVisionModel.for_inference(model)

def _norm(s): return re.sub(r'\s+', ' ', re.sub(r'[.,]', '', (s or '').strip().lower()))

def _gen(ex):
    img = Image.open(ex['image_path']).convert('RGB')
    msgs = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user',   'content': [
            {'type': 'image', 'image': img},
            {'type': 'text',  'text': f"Beverage type: {ex['beverage_type']}. Extract per the schema."},
        ]},
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, images=img, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    txt = tokenizer.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0]
    try:
        s, e = txt.find('{'), txt.rfind('}')
        return json.loads(txt[s:e+1])
    except Exception: return None

from tqdm.auto import tqdm
_run_eval(test, _gen)  # defined in next cell


## 10. Save adapter to Drive + zip + auto-download to Mac


In [ ]:
import shutil, json
ADAPTER_DIR = WORK_DIR / 'adapter'; ADAPTER_DIR.mkdir(exist_ok=True)

model.save_pretrained(str(ADAPTER_DIR))
try: tokenizer.save_pretrained(str(ADAPTER_DIR))
except NameError: processor.save_pretrained(str(ADAPTER_DIR))

manifest = {'model_tag': MODEL_TAG, 'base_model': BASE_MODEL,
            'trained_on': f'COLA Cloud free sample N={len(train)}',
            'lora_rank': 16, 'epochs': 2}
with open(ADAPTER_DIR / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'Adapter → {ADAPTER_DIR}')

# Bundle + browser-download
BUNDLE = WORK_DIR / f'ttb_sft_{MODEL_TAG}_bundle'
if BUNDLE.exists(): shutil.rmtree(BUNDLE)
BUNDLE.mkdir()
for src in ['adapter', 'splits', 'eval_report.json']:
    s = WORK_DIR / src
    if not s.exists(): continue
    d = BUNDLE / src
    (shutil.copytree if s.is_dir() else shutil.copy)(s, d)

zip_path = shutil.make_archive(str(WORK_DIR / f'ttb_sft_{MODEL_TAG}'), 'zip', root_dir=str(BUNDLE))
import os; size_mb = os.path.getsize(zip_path) / 1024**2
print(f'Bundle: {zip_path}  ({size_mb:.1f} MB)')

from google.colab import files
files.download(zip_path)
print('Browser download started — accept the prompt.')


## 11. Next: integrate the adapter into the backend

On your Mac:
```
unzip ~/Downloads/ttb_sft_{model_tag}.zip -d backend/models/{model_tag}/
```

We'll add `backend/app/extractors/sft.py` (loads base model + LoRA via vLLM or transformers) and register `INFERENCE_MODE=sft` in the factory. Then
`make eval-real` does the head-to-head against Claude.
